# Action Baseline Walkthrough

A reading companion for the CLI and webpage that runs entirely from the **committed** `outputs/` artifacts — no raw Xperience-10M data needed. To regenerate the artifacts from raw data, set `DATA_ROOT` and run `make sample` / `make split-comparison` / `make dino`.

It walks the three findings the repo is built around: the split decides the number, hands beat pixels, and fusion is not free.

In [1]:
import json
from pathlib import Path

REPO = Path.cwd()
while not (REPO / "outputs").exists() and REPO != REPO.parent:
    REPO = REPO.parent

def load(rel):
    return json.loads((REPO / rel).read_text())

summary = load("outputs/sample_ablation/summary.json")
print(summary["num_windows"], "windows,", len(summary["classes"]), "action classes, split:", summary["split_strategy"])

1093 windows, 18 action classes, split: blocked-instance


## 1. The split decides the number

The same `hand_joints_only` model, evaluated three ways. The spread is two orders of magnitude — proof that evaluation design, not the model, drives the headline on a single episode.

In [2]:
strat = load("outputs/split_comparison/stratified/summary.json")
chrono = load("outputs/split_comparison/chronological/summary.json")
print("stratified (leaky):          %.3f" % strat["experiments"]["hand_joints_only"]["accuracy"])
print("chronological (label shift): %.3f" % chrono["experiments"]["hand_joints_only"]["accuracy"])
print("blocked-instance (honest):   %.3f" % summary["experiments"]["hand_joints_only"]["accuracy"])

stratified (leaky):          0.941
chronological (label shift): 0.004
blocked-instance (honest):   0.471


## 2. Hands beat pixels, and fusion is not free

Under the honest blocked-instance split: hand motion is the strongest single cue, and every fusion strategy lands *between* its inputs rather than above the best one.

In [3]:
order = ["rgb_only_majority", "rgb_only", "hand_joints_only", "rgb_hand_fusion", "rgb_hand_late_fusion", "rgb_hand_gated_fusion"]
for name in order:
    m = summary["experiments"].get(name)
    if m:
        gate = ("  gate=%.2f" % m["mean_rgb_gate"]) if "mean_rgb_gate" in m else ""
        print("%-24s acc=%.3f  macroF1=%.3f%s" % (name, m["accuracy"], m["macro_f1"], gate))

rgb_only_majority        acc=0.143  macroF1=0.015
rgb_only                 acc=0.257  macroF1=0.162
hand_joints_only         acc=0.471  macroF1=0.259
rgb_hand_fusion          acc=0.357  macroF1=0.262
rgb_hand_late_fusion     acc=0.327  macroF1=0.190
rgb_hand_gated_fusion    acc=0.349  macroF1=0.249  gate=0.51


## 3. A stronger representation helps but does not change the verdict

Swapping handcrafted RGB for frozen DINOv2 embeddings (same head, same split) lifts every RGB-based number — yet hands alone still win.

In [4]:
dino = load("outputs/dino_ablation/summary.json")
for name in ["rgb_only", "rgb_hand_fusion", "rgb_hand_late_fusion"]:
    print("%-22s handcrafted=%.3f  dino=%.3f" % (name, summary["experiments"][name]["accuracy"], dino["experiments"][name]["accuracy"]))
print("hand_joints_only (unchanged): %.3f" % summary["experiments"]["hand_joints_only"]["accuracy"])

rgb_only               handcrafted=0.257  dino=0.312
rgb_hand_fusion        handcrafted=0.357  dino=0.408
rgb_hand_late_fusion   handcrafted=0.327  dino=0.393
hand_joints_only (unchanged): 0.471


## 4. The errors are about objects, not motions

The largest confusions are verbs that share an object — the difficulty is verb granularity, an egocentric hallmark.

In [5]:
for c in summary["experiments"]["hand_joints_only"]["top_confusions"]:
    print("%-26s -> %-26s %d (%.0f%% of true)" % (c["true"], c["predicted"], c["count"], 100 * c["fraction_of_true"]))

Grasp coffee scoop         -> Transfer coffee to dripper 17 (100% of true)
Position kettle to pour    -> Move kettle away           16 (52% of true)
Hold gooseneck kettle      -> Grasp gooseneck kettle     12 (31% of true)


## How to read the result

Accuracy is overall correctness; macro F1 weights each action class equally, which matters on imbalanced episodes; ECE (in each `metrics.json`) checks whether confidence is trustworthy. None of these numbers prove cross-episode generalization — for that, add episodes and use the `grouped-segment` split.